# Compare source ROOT files: exp vs reco
# Goal: understand why events from exp.h5 don't match exp_reco.h5
# by comparing the raw ROOT files they were built from.

In [1]:
import numpy as np
import uproot as ur
import os
import re
from collections import defaultdict

In [2]:
EXP_ROOT_DIR = "/net/62/home/albert/Baikal/Data/exp_root_files/"
RECO_ROOT_DIR = "/net/62/home/albert/Baikal/Data/reco_exp_root_files/"

# Branches to read
PULSE_N = "Events/BEvent./BEvent.fPulseN"
PULSE_AMP = "Events/BEvent./BEvent.fPulses/BEvent.fPulses.fAmplitude"
PULSE_TIME = "Events/BEvent./BEvent.fPulses/BEvent.fPulses.fTime"
PULSE_CHID = "Events/BEvent./BEvent.fPulses/BEvent.fPulses.fChannelID"

## Build matched file pairs (exp ↔ reco)

In [3]:
def parse_exp_filename(fname: str) -> tuple:
    """'s2020_c01_r0027.root' → (2020, 1, 27)"""
    m = re.match(r's(\d+)_c(\d+)_r(\d+)\.root', fname)
    return (int(m.group(1)), int(m.group(2)), int(m.group(3))) if m else None

def parse_reco_filename(fname: str) -> tuple:
    """'2020_cl1_run27_scl_nu_DATA2020.root' → (2020, 1, 27)"""
    m = re.match(r'(\d+)_cl(\d+)_run(\d+)', fname)
    return (int(m.group(1)), int(m.group(2)), int(m.group(3))) if m else None

# Match files by (year, cluster, run)
exp_files = {parse_exp_filename(f): f for f in os.listdir(EXP_ROOT_DIR) if f.endswith('.root')}
reco_files = {parse_reco_filename(f): f for f in os.listdir(RECO_ROOT_DIR) if f.endswith('.root')}

common_keys = sorted(set(exp_files) & set(reco_files))
print(f"Exp files: {len(exp_files)}, Reco files: {len(reco_files)}, Matched: {len(common_keys)}")

exp_only = set(exp_files) - set(reco_files)
reco_only = set(reco_files) - set(exp_files)
if exp_only:
    print(f"Exp-only runs: {sorted(exp_only)}")
if reco_only:
    print(f"Reco-only runs: {sorted(reco_only)}")

Exp files: 26, Reco files: 26, Matched: 26


## Compare event counts and hit data for each matched pair

In [4]:
def get_start_index(rf) -> int:
    """First event with PulseN > 0."""
    pulses_nums = rf[PULSE_N].array(library="np")
    mask = pulses_nums > 0
    return int(np.argwhere(mask)[0][0]) if np.any(mask) else len(pulses_nums)

def read_root_summary(rf_path: str) -> dict:
    """Read basic event/hit info from a ROOT file."""
    with ur.open(rf_path) as rf:
        st = get_start_index(rf)
        total_entries = rf[PULSE_N].num_entries
        pulse_n = rf[PULSE_N].array(library="np")[st:]
        return {
            "total_entries": total_entries,
            "start_index": st,
            "n_events": len(pulse_n),
            "n_hits_total": int(pulse_n.sum()),
            "pulse_n": pulse_n,
        }

In [5]:
results = []
for key in common_keys:
    exp_path = os.path.join(EXP_ROOT_DIR, exp_files[key])
    reco_path = os.path.join(RECO_ROOT_DIR, reco_files[key])

    exp_info = read_root_summary(exp_path)
    reco_info = read_root_summary(reco_path)

    # Per-event hit count comparison (for events present in both)
    n_common = min(exp_info["n_events"], reco_info["n_events"])
    hit_match = np.array_equal(exp_info["pulse_n"][:n_common], reco_info["pulse_n"][:n_common])

    results.append({
        "run": key,
        "exp_file": exp_files[key],
        "reco_file": reco_files[key],
        "exp_entries": exp_info["total_entries"],
        "reco_entries": reco_info["total_entries"],
        "exp_st": exp_info["start_index"],
        "reco_st": reco_info["start_index"],
        "exp_events": exp_info["n_events"],
        "reco_events": reco_info["n_events"],
        "exp_hits": exp_info["n_hits_total"],
        "reco_hits": reco_info["n_hits_total"],
        "hit_match_first_n": hit_match,
        "exp_pulse_n": exp_info["pulse_n"],
        "reco_pulse_n": reco_info["pulse_n"],
    })

    status = "OK" if (exp_info["n_events"] == reco_info["n_events"] and hit_match) else "DIFF"
    print(
        f"{status} | {key} | "
        f"entries: {exp_info['total_entries']}/{reco_info['total_entries']} | "
        f"st: {exp_info['start_index']}/{reco_info['start_index']} | "
        f"events: {exp_info['n_events']}/{reco_info['n_events']} | "
        f"hits: {exp_info['n_hits_total']}/{reco_info['n_hits_total']}"
    )

DIFF | (2020, 1, 27) | entries: 25000/11794 | st: 0/1 | events: 25000/11793 | hits: 1372329/844731
DIFF | (2020, 1, 116) | entries: 25000/27448 | st: 0/1 | events: 25000/27447 | hits: 1313631/1927669
DIFF | (2020, 1, 188) | entries: 25000/27401 | st: 0/1 | events: 25000/27400 | hits: 1741114/2334596
DIFF | (2020, 2, 15) | entries: 25000/6871 | st: 0/1 | events: 25000/6870 | hits: 1359146/474979
DIFF | (2020, 2, 149) | entries: 25000/524 | st: 0/1 | events: 25000/523 | hits: 1656530/43307
DIFF | (2020, 2, 206) | entries: 25000/461 | st: 0/1 | events: 25000/460 | hits: 1803293/41343
DIFF | (2020, 2, 357) | entries: 25000/215 | st: 0/1 | events: 25000/214 | hits: 1971454/20600
DIFF | (2020, 3, 12) | entries: 25000/10868 | st: 0/1 | events: 25000/10867 | hits: 1437882/789863
DIFF | (2020, 3, 115) | entries: 25000/2927 | st: 0/1 | events: 25000/2926 | hits: 1444261/214779
DIFF | (2020, 3, 263) | entries: 25000/46011 | st: 0/1 | events: 25000/46010 | hits: 1823468/4052129
DIFF | (2020, 3, 41

## Detailed analysis of mismatched runs

In [ ]:
for r in results:
    exp_pn = r["exp_pulse_n"]
    reco_pn = r["reco_pulse_n"]

    if r["exp_events"] == r["reco_events"] and r["hit_match_first_n"]:
        continue  # skip matching runs

    print(f"\n{'='*60}")
    print(f"Run: {r['run']}  |  exp: {r['exp_file']}  |  reco: {r['reco_file']}")
    print(f"  Total entries:  exp={r['exp_entries']}, reco={r['reco_entries']}")
    print(f"  Start index:    exp={r['exp_st']}, reco={r['reco_st']}")
    print(f"  Events:         exp={r['exp_events']}, reco={r['reco_events']}")
    print(f"  Total hits:     exp={r['exp_hits']}, reco={r['reco_hits']}")

    # Check if reco is a subset of exp (reco has fewer events = filtered)
    if r["reco_events"] < r["exp_events"]:
        # Try to find reco events inside exp by matching pulse_n sequences
        # Simple check: does reco_pulse_n appear as a contiguous block in exp_pulse_n?
        found_at = None
        for offset in range(r["exp_events"] - r["reco_events"] + 1):
            if np.array_equal(exp_pn[offset:offset + len(reco_pn)], reco_pn):
                found_at = offset
                break
        if found_at is not None:
            print(f"  → Reco pulse_n is a contiguous subsequence of exp starting at event {found_at}")
        else:
            print(f"  → Reco pulse_n is NOT a contiguous subsequence of exp")

    elif r["reco_events"] == r["exp_events"]:
        # Same count but different hits — find mismatches
        mismatch = exp_pn != reco_pn
        n_mismatch = mismatch.sum()
        print(f"  → Same event count, but {n_mismatch} events have different hit counts")
        if n_mismatch > 0 and n_mismatch <= 20:
            idx = np.where(mismatch)[0]
            for i in idx:
                print(f"    Event {i}: exp={exp_pn[i]}, reco={reco_pn[i]}")

    # Compare per-event hit counts for shared prefix
    n_common = min(len(exp_pn), len(reco_pn))
    prefix_match = np.array_equal(exp_pn[:n_common], reco_pn[:n_common])
    if not prefix_match:
        mismatched = exp_pn[:n_common] != reco_pn[:n_common]
        first_diff = np.argwhere(mismatched)[0][0]
        print(f"  → First hit-count mismatch at event {first_diff}: exp={exp_pn[first_diff]}, reco={reco_pn[first_diff]}")
        print(f"  → Total mismatched events in shared range: {mismatched.sum()}/{n_common}")